In [2]:
import re

def parse_gml_id_label(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        txt = f.read()

    # node [ id X label "...." ] を抜く
    # （nodeブロックは他の属性もあるが、idとlabelだけ取ればよい）
    pairs = re.findall(r'node\s*\[\s*id\s+(\d+)\s*label\s+"([^"]+)"', txt)
    return {int(i): lab for i, lab in pairs}


In [3]:
import networkx as nx

def parse_gml_edges(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        txt = f.read()

    # edge [ source X target Y ] を抽出
    edges = re.findall(r"edge\s*\[\s*source\s+(\d+)\s*target\s+(\d+)", txt)
    return [(int(a), int(b)) for a, b in edges]

def load_polblogs_gml_dedup(path):
    id2label = parse_gml_id_label(path)
    edges = parse_gml_edges(path)

    # 有向グラフとしてまず作る（gmlは directed 1）
    G = nx.DiGraph()

    # ノードは全IDを登録
    all_ids = set(id2label.keys())
    for u, v in edges:
        all_ids.add(u)
        all_ids.add(v)
    G.add_nodes_from(all_ids)

    # エッジは重複排除して追加
    G.add_edges_from(set(edges))

    # URL(label) へリラベル（比較用）
    G_url = nx.relabel_nodes(G, lambda i: id2label.get(i, str(i)))

    return G_url, id2label

In [4]:
G_gml_url, id2label = load_polblogs_gml_dedup("polblogs.gml")
print("GML loaded:", G_gml_url.number_of_nodes(), G_gml_url.number_of_edges())

GML loaded: 1490 19025


In [6]:
import networkx as nx
G_gexf = nx.read_gexf("polblogs_rf_sinkhorn_e2_20.gexf")
print("GEXF loaded:", G_gexf.number_of_nodes(), G_gexf.number_of_edges())

GEXF loaded: 1222 16714


In [7]:
G_gml_u  = G_gml_url.to_undirected()
G_gexf_u = G_gexf.to_undirected()

nodes_gml  = set(G_gml_u.nodes())
nodes_gexf = set(G_gexf_u.nodes())

print("nodes only in gml :", len(nodes_gml - nodes_gexf))
print("nodes only in gexf:", len(nodes_gexf - nodes_gml))

edges_gml  = {frozenset(e) for e in G_gml_u.edges()}
edges_gexf = {frozenset(e) for e in G_gexf_u.edges()}

print("edges only in gml :", len(edges_gml - edges_gexf))
print("edges only in gexf:", len(edges_gexf - edges_gml))


nodes only in gml : 269
nodes only in gexf: 1
edges only in gml : 5
edges only in gexf: 1


In [8]:
from networkx.algorithms.graph_hashing import weisfeiler_lehman_graph_hash

h1 = weisfeiler_lehman_graph_hash(G_gml_u)
h2 = weisfeiler_lehman_graph_hash(G_gexf_u)
print("WL hash same?", h1 == h2)


WL hash same? False


In [9]:
import networkx as nx

def lcc(G):
    comps = list(nx.connected_components(G))
    comps.sort(key=len, reverse=True)
    return G.subgraph(comps[0]).copy()

G_gml_lcc = lcc(G_gml_u)

print("GML LCC:", G_gml_lcc.number_of_nodes(), G_gml_lcc.number_of_edges())
print("GEXF    :", G_gexf_u.number_of_nodes(), G_gexf_u.number_of_edges())

GML LCC: 1222 16717
GEXF    : 1222 16714


In [10]:
# gml側：削除されたノード集合
missing_nodes = set(G_gml_u.nodes()) - set(G_gexf_u.nodes())

# gml側の "source" 属性を取る（あなたの parse で source も拾っているならそれを使う）
# もし拾っていない場合は、gmlのテキストから source もパースする必要あり

sources = nx.get_node_attributes(G_gml_u, "source")
from collections import Counter
print(Counter(sources.get(n, "NA") for n in missing_nodes).most_common(20))


[('NA', 269)]


In [11]:
common_nodes = set(G_gml_u.nodes()) & set(G_gexf_u.nodes())

G_gml_common  = G_gml_u.subgraph(common_nodes).copy()
G_gexf_common = G_gexf_u.subgraph(common_nodes).copy()

print(G_gml_common.number_of_nodes(), G_gml_common.number_of_edges())
print(G_gexf_common.number_of_nodes(), G_gexf_common.number_of_edges())


1221 16716
1221 16713


In [12]:
from networkx.algorithms.graph_hashing import weisfeiler_lehman_graph_hash
print(weisfeiler_lehman_graph_hash(G_gml_common) == weisfeiler_lehman_graph_hash(G_gexf_common))


False


In [13]:
G_gml_lcc = lcc(G_gml_u)
print("GML LCC:", G_gml_lcc.number_of_nodes(), G_gml_lcc.number_of_edges())
print("GEXF   :", G_gexf_u.number_of_nodes(), G_gexf_u.number_of_edges())


GML LCC: 1222 16717
GEXF   : 1222 16714
